# Preprocess And Align With LightGlue

This notebook converts raw uploads into processed JPEGs, builds `img_labels.csv`, then aligns each location group with SuperPoint + LightGlue. Use `REDO_PREPROCESS` and `REDO_ALIGNMENT` independently to rebuild either stage from scratch.

In [ ]:
# Colab/runtime setup.
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

!pip install -q uv
!uv pip install --system git+https://github.com/cvg/LightGlue.git pillow pillow-heif pandas opencv-python-headless torch torchvision matplotlib tqdm

In [ ]:
import gc
import json
import os
import re
import shutil
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageOps, UnidentifiedImageError
from pillow_heif import register_heif_opener
from lightglue import LightGlue, SuperPoint
from lightglue.utils import load_image
from tqdm.auto import tqdm

register_heif_opener()
plt.rcParams['figure.figsize'] = (14, 7)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

def bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def clean_stem(value):
    stem = Path(str(value)).stem.lower().strip()
    stem = re.sub(r'\s+', '_', stem)
    stem = re.sub(r'[^a-z0-9_()\-]+', '_', stem)
    return stem.strip('_') or 'image'

def ensure_empty_dir(path):
    if os.path.isdir(path):
        shutil.rmtree(path)
    os.makedirs(path, exist_ok=True)

def file_exists_and_nonempty(path):
    return os.path.exists(path) and os.path.getsize(path) > 0

In [ ]:
# Main settings.
BASE_CANDIDATES = [
    '/content/drive/MyDrive/CIS_5190_group_project',
    '/content/drive/My Drive/CIS_5190_group_project',
    os.path.abspath('../data'),
]
BASE = next((p for p in BASE_CANDIDATES if os.path.exists(p)), BASE_CANDIDATES[0])

RAW_DIR = os.path.join(BASE, 'Images')
PROCESSED_DIR = os.path.join(BASE, 'processedImages') if 'content' in BASE else os.path.join(BASE, 'processed')
LABELS_CSV = os.path.join(BASE, 'img_labels.csv')
ALIGNED_DIR = os.path.join(BASE, 'aligned_lightglue')
ALIGNED_CSV = os.path.join(BASE, 'aligned_lightglue_labels.csv')
FILTERED_DIR = os.path.join(BASE, 'filtered_aligned_lightglue')
FILTERED_CSV = os.path.join(BASE, 'filtered_aligned_lightglue_labels.csv')

# Defaults are True so newly uploaded/replaced images are always swept into the pipeline.
REDO_PREPROCESS = True
REDO_ALIGNMENT = True
REDO_FILTERED_ALIGNMENT = True

# Use a list like ['34th', 'agh3rd'] while debugging alignment, or None for every location.
LOCATION_FILTER = None

# LightGlue / homography settings.
RESIZE = 1024
MAX_KEYPOINTS = 2048
RANSAC_REPROJ_THRESHOLD = 5.0
MIN_MATCHES = 12
MIN_INLIERS = 8
MIN_INLIER_RATIO = 0.0

# Duplicate filtering settings. Used after alignment to keep one image per location/time/weather.
ECC_SCORE_SIZE = 768
ECC_MAX_ITERS = 80
ECC_EPS = 1e-5
ECC_MAX_WORSE_FACTOR = 1.05
ECC_MAX_CORNER_DRIFT_FRAC = 0.20
ECC_MIN_SCALE = 0.70
ECC_MAX_SCALE = 1.30

# Output settings. Use 512 to match the old pipeline; set None to keep shared crop native size.
OUTPUT_SIZE = 512
JPEG_QUALITY = 95

print('Base:', BASE)
print('Raw images:', RAW_DIR)
print('Processed images:', PROCESSED_DIR)
print('Labels CSV:', LABELS_CSV)
print('Aligned images:', ALIGNED_DIR)
print('Aligned CSV:', ALIGNED_CSV)
print('Filtered aligned images:', FILTERED_DIR)
print('Filtered aligned CSV:', FILTERED_CSV)

## Preprocess Raw Images

If `REDO_PREPROCESS = False`, this stage skips only when `img_labels.csv` exists and every `file_name` listed in it exists under `processedImages/`. If `REDO_PREPROCESS = True`, it deletes and rebuilds `processedImages/` and `img_labels.csv`.

In [ ]:
VALID_IMAGE_SUFFIXES = {'.heic', '.heif', '.jpg', '.jpeg', '.png'}
FILENAME_PATTERN = re.compile(r'^([^_]+)_([^_]+)_([a-zA-Z]+)')

def parse_filename(fname):
    match = FILENAME_PATTERN.match(fname)
    if not match:
        return None
    location, tod, weather = match.groups()
    return location.lower(), tod.lower(), weather.lower(), clean_stem(fname)

def preprocess_complete():
    if not file_exists_and_nonempty(LABELS_CSV):
        return False
    try:
        df = pd.read_csv(LABELS_CSV)
    except Exception:
        return False
    required = {'original_file_name', 'file_name', 'location', 'location_index', 'time_of_day', 'weather', 'condition_dir'}
    if not required.issubset(df.columns) or df.empty:
        return False
    missing = [p for p in df['file_name'].astype(str) if not os.path.exists(os.path.join(PROCESSED_DIR, p))]
    if missing:
        print('Processed files missing from existing CSV:', missing[:5])
        return False
    return True

def run_preprocess():
    assert os.path.isdir(RAW_DIR), f'Missing raw image directory: {RAW_DIR}'
    if REDO_PREPROCESS:
        print('REDO_PREPROCESS=True; deleting previous processed images and labels CSV.')
        ensure_empty_dir(PROCESSED_DIR)
        if os.path.exists(LABELS_CSV):
            os.remove(LABELS_CSV)
    elif preprocess_complete():
        print('Preprocess already complete; skipping.')
        return pd.read_csv(LABELS_CSV)
    else:
        os.makedirs(PROCESSED_DIR, exist_ok=True)

    parsed_files = []
    skipped_non_images = []
    badly_named = []
    skipped_unreadable = []

    for fname in sorted(os.listdir(RAW_DIR)):
        suffix = os.path.splitext(fname)[1].lower()
        if ':zone.identifier' in fname.lower() or suffix not in VALID_IMAGE_SUFFIXES:
            skipped_non_images.append(fname)
            continue
        parsed = parse_filename(fname)
        if parsed is None:
            badly_named.append(fname)
            continue
        parsed_files.append((fname, *parsed))

    location_to_index = {loc: idx for idx, loc in enumerate(sorted({row[1] for row in parsed_files}))}
    records = []

    for fname, location, tod, weather, stem in tqdm(parsed_files, desc='Preprocess images'):
        in_path = os.path.join(RAW_DIR, fname)
        location_index = location_to_index[location]
        condition_dir = f'{tod}_{weather}'
        out_name = f'{stem}.jpg'
        rel_path = os.path.join(str(location_index), condition_dir, out_name).lower()
        out_path = os.path.join(PROCESSED_DIR, rel_path)
        os.makedirs(os.path.dirname(out_path), exist_ok=True)

        try:
            img = ImageOps.exif_transpose(Image.open(in_path)).convert('RGB')
            img.save(out_path, 'JPEG', quality=JPEG_QUALITY)
        except (UnidentifiedImageError, OSError) as exc:
            skipped_unreadable.append((fname, str(exc)))
            continue

        records.append({
            'original_file_name': fname.lower(),
            'file_name': rel_path,
            'location': location,
            'location_index': location_index,
            'time_of_day': tod,
            'weather': weather,
            'condition_dir': condition_dir.lower(),
        })

    df = pd.DataFrame(records).sort_values(['location_index', 'condition_dir', 'file_name'])
    df.to_csv(LABELS_CSV, index=False)
    print(f'Saved {len(df)} rows to {LABELS_CSV}')
    if badly_named:
        print(f'Badly named files ({len(badly_named)}):', badly_named[:20])
    if skipped_non_images:
        print(f'Skipped non-image sidecar/files ({len(skipped_non_images)}):', skipped_non_images[:20])
    if skipped_unreadable:
        print(f'Skipped unreadable image files ({len(skipped_unreadable)}):', skipped_unreadable[:10])
    return df

labels_df = run_preprocess()
display(labels_df.head())

## Align Processed Images

If `REDO_ALIGNMENT = False`, this stage skips only when `aligned_lightglue_labels.csv` exists and every saved `aligned_file` exists under `aligned_lightglue/`. If `REDO_ALIGNMENT = True`, it deletes and rebuilds the aligned directory and CSV.

In [ ]:
DAYLIKE = {'daytime', 'day', 'morning'}

def alignment_complete():
    if not file_exists_and_nonempty(ALIGNED_CSV):
        return False
    try:
        df = pd.read_csv(ALIGNED_CSV)
    except Exception:
        return False
    if df.empty or 'aligned_file' not in df.columns:
        return False
    missing = [p for p in df['aligned_file'].astype(str) if not os.path.exists(os.path.join(ALIGNED_DIR, p))]
    if missing:
        print('Aligned files missing from existing CSV:', missing[:5])
        return False
    return True

def pick_anchor(group):
    tod = group['time_of_day'].astype(str).str.lower().str.strip()
    weather = group['weather'].astype(str).str.lower().str.strip()
    candidates = group[tod.isin(DAYLIKE) & (weather == 'clear')]
    if candidates.empty:
        candidates = group[tod.isin(DAYLIKE)]
    if candidates.empty:
        candidates = group
    return candidates.iloc[0]

def load_bgr_checked(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)
    if img is None:
        raise FileNotFoundError(path)
    return img

def lightglue_match(anchor_path, target_path):
    image0_raw = load_image(anchor_path, resize=RESIZE)
    image1_raw = load_image(target_path, resize=RESIZE)
    image0 = image0_raw.mean(dim=0, keepdim=True).unsqueeze(0).to(device)
    image1 = image1_raw.mean(dim=0, keepdim=True).unsqueeze(0).to(device)

    with torch.inference_mode():
        feats0 = extractor({'image': image0})
        feats1 = extractor({'image': image1})
        matches01 = matcher({'image0': feats0, 'image1': feats1})

    kpts0 = feats0['keypoints'][0].detach().cpu().numpy()
    kpts1 = feats1['keypoints'][0].detach().cpu().numpy()
    matches = matches01['matches'][0].detach().cpu().numpy()
    if len(matches) == 0:
        return None, {'matches': 0, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'no_matches'}
    return (image0_raw, image1_raw, kpts0[matches[:, 0]], kpts1[matches[:, 1]], len(matches)), None

def estimate_target_to_anchor(anchor_bgr, target_bgr, anchor_path, target_path):
    match_result, error = lightglue_match(anchor_path, target_path)
    if error is not None:
        return None, error

    image0_raw, image1_raw, mkpts0, mkpts1, n_matches = match_result
    if n_matches < MIN_MATCHES:
        return None, {'matches': n_matches, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'not_enough_matches'}

    h0, w0 = anchor_bgr.shape[:2]
    h1, w1 = target_bgr.shape[:2]
    scale0 = np.array([w0 / image0_raw.shape[2], h0 / image0_raw.shape[1]])
    scale1 = np.array([w1 / image1_raw.shape[2], h1 / image1_raw.shape[1]])
    mkpts0_orig = mkpts0 * scale0
    mkpts1_orig = mkpts1 * scale1

    H, inlier_mask = cv2.findHomography(mkpts1_orig, mkpts0_orig, cv2.USAC_MAGSAC, RANSAC_REPROJ_THRESHOLD)
    if H is None or inlier_mask is None:
        return None, {'matches': n_matches, 'inliers': 0, 'inlier_ratio': 0.0, 'reason': 'homography_failed'}

    inliers = int(inlier_mask.ravel().sum())
    inlier_ratio = inliers / max(n_matches, 1)
    if inliers < MIN_INLIERS:
        return None, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'too_few_inliers'}
    if MIN_INLIER_RATIO > 0 and inlier_ratio < MIN_INLIER_RATIO:
        return None, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'too_low_inlier_ratio'}

    return H, {'matches': n_matches, 'inliers': inliers, 'inlier_ratio': inlier_ratio, 'reason': 'ok'}

def resize_output(img):
    if OUTPUT_SIZE is None:
        return img
    return cv2.resize(img, (OUTPUT_SIZE, OUTPUT_SIZE), interpolation=cv2.INTER_AREA)

def unique_output_name(location_dir, stem):
    base = f'{stem}_aligned.jpg'
    candidate = base
    counter = 2
    while os.path.exists(os.path.join(location_dir, candidate)):
        candidate = f'{stem}_aligned_{counter}.jpg'
        counter += 1
    return candidate

In [ ]:
def align_location_group(location, group):
    group = group.copy().reset_index(drop=True)
    anchor_row = pick_anchor(group)
    anchor_file = anchor_row['file_name']
    anchor_path = os.path.join(PROCESSED_DIR, anchor_file)

    try:
        anchor_bgr = load_bgr_checked(anchor_path)
    except FileNotFoundError:
        print(f'[SKIP] {location}: missing anchor {anchor_path}')
        return []

    h0, w0 = anchor_bgr.shape[:2]
    aligned_items = [{
        'row': anchor_row,
        'image': anchor_bgr,
        'mask': np.ones((h0, w0), dtype=np.uint8) * 255,
        'stats': {'matches': 0, 'inliers': 0, 'inlier_ratio': 1.0, 'reason': 'anchor'},
        'homography': np.eye(3),
        'is_anchor': True,
    }]

    print(f'\nProcessing {location}: {len(group)} images | anchor={anchor_file}')
    for _, row in group.iterrows():
        if row['file_name'] == anchor_file:
            continue

        target_file = row['file_name']
        target_path = os.path.join(PROCESSED_DIR, target_file)
        try:
            target_bgr = load_bgr_checked(target_path)
        except FileNotFoundError:
            print(f'  [MISS] {target_file}')
            continue

        H, stats = estimate_target_to_anchor(anchor_bgr, target_bgr, anchor_path, target_path)
        if H is None:
            print(f"  [FAIL] {target_file} matches={stats['matches']} inliers={stats['inliers']} ratio={stats['inlier_ratio']:.2f} reason={stats['reason']}")
            continue

        aligned = cv2.warpPerspective(target_bgr, H, (w0, h0))
        source_mask = np.ones(target_bgr.shape[:2], dtype=np.uint8) * 255
        warped_mask = cv2.warpPerspective(source_mask, H, (w0, h0))
        aligned_items.append({
            'row': row,
            'image': aligned,
            'mask': warped_mask,
            'stats': stats,
            'homography': H,
            'is_anchor': False,
        })
        print(f"  [OK] {target_file} matches={stats['matches']} inliers={stats['inliers']} ratio={stats['inlier_ratio']:.2f}")

    shared_mask = aligned_items[0]['mask']
    for item in aligned_items[1:]:
        shared_mask = cv2.bitwise_and(shared_mask, item['mask'])
    coords = cv2.findNonZero(shared_mask)
    if coords is None:
        print(f'  [SKIP] {location}: no shared crop after alignment')
        return []

    x, y, crop_w, crop_h = cv2.boundingRect(coords)
    location_dir = os.path.join(ALIGNED_DIR, clean_stem(location))
    os.makedirs(location_dir, exist_ok=True)

    records = []
    for item in aligned_items:
        row = item['row']
        cropped = resize_output(item['image'][y:y + crop_h, x:x + crop_w])
        out_name = unique_output_name(location_dir, clean_stem(row.get('original_file_name', row['file_name'])))
        out_path = os.path.join(location_dir, out_name)
        cv2.imwrite(out_path, cropped, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

        stats = item['stats']
        records.append({
            'location': location,
            'source_file': row['file_name'],
            'original_file_name': row.get('original_file_name', ''),
            'aligned_file': os.path.relpath(out_path, ALIGNED_DIR),
            'is_anchor': item['is_anchor'],
            'anchor_file': anchor_file,
            'time_of_day': row.get('time_of_day', ''),
            'weather': row.get('weather', ''),
            'matches': stats['matches'],
            'inliers': stats['inliers'],
            'inlier_ratio': round(float(stats['inlier_ratio']), 4),
            'crop_x': x,
            'crop_y': y,
            'crop_w': crop_w,
            'crop_h': crop_h,
            'output_size': OUTPUT_SIZE or '',
            'homography_json': json.dumps(item['homography'].tolist()),
        })

    print(f'  [SAVE] {len(records)} cropped aligned images | crop=({x}, {y}, {crop_w}, {crop_h})')
    return records

In [ ]:
def run_alignment(labels_df):
    if REDO_ALIGNMENT:
        print('REDO_ALIGNMENT=True; deleting previous aligned images and aligned CSV.')
        ensure_empty_dir(ALIGNED_DIR)
        if os.path.exists(ALIGNED_CSV):
            os.remove(ALIGNED_CSV)
    elif alignment_complete():
        print('Alignment already complete; skipping.')
        return pd.read_csv(ALIGNED_CSV)
    else:
        os.makedirs(ALIGNED_DIR, exist_ok=True)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    global extractor, matcher
    extractor = SuperPoint(max_num_keypoints=MAX_KEYPOINTS).eval().to(device)
    matcher = LightGlue(features='superpoint').eval().to(device)

    df = labels_df.copy()
    required_cols = {'file_name', 'location', 'time_of_day', 'weather'}
    missing_cols = required_cols - set(df.columns)
    assert not missing_cols, f'Missing columns: {missing_cols}'
    df['file_name'] = df['file_name'].astype(str)
    df['location'] = df['location'].astype(str).str.lower().str.strip()
    df['time_of_day'] = df['time_of_day'].astype(str).str.lower().str.strip()
    df['weather'] = df['weather'].astype(str).str.lower().str.strip()

    if LOCATION_FILTER is not None:
        wanted = {str(x).lower().strip() for x in LOCATION_FILTER}
        df = df[df['location'].isin(wanted)].copy()

    all_records = []
    for location, group in tqdm(list(df.groupby('location', sort=True)), desc='Align locations'):
        all_records.extend(align_location_group(location, group))

    out_df = pd.DataFrame(all_records)
    out_df.to_csv(ALIGNED_CSV, index=False)
    print(f'\nDone. Saved {len(out_df)} rows to {ALIGNED_CSV}')
    return out_df

aligned_df = run_alignment(labels_df)
display(aligned_df.head())

## Filter Duplicate Conditions

This stage keeps one representative image for each unique `location + time_of_day + weather`. When there are multiple copies of the same condition, it scores each candidate against the other copies after a small extra ECC homography alignment and keeps the image with the best average match score.

In [ ]:
def filtered_complete():
    if not file_exists_and_nonempty(FILTERED_CSV):
        return False
    try:
        df = pd.read_csv(FILTERED_CSV)
    except Exception:
        return False
    if df.empty or 'filtered_file' not in df.columns:
        return False
    missing = [p for p in df['filtered_file'].astype(str) if not os.path.exists(os.path.join(FILTERED_DIR, p))]
    if missing:
        print('Filtered files missing from existing CSV:', missing[:5])
        return False
    return True

def prepare_ecc_gray(img):
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h, w = gray.shape[:2]
    scale = min(1.0, ECC_SCORE_SIZE / max(h, w))
    if scale < 1.0:
        gray = cv2.resize(gray, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    gray = gray.astype(np.float32) / 255.0
    return gray

def ecc_warp_is_sane(warp, shape):
    h, w = shape[:2]
    corners = np.float32([[0, 0], [w - 1, 0], [w - 1, h - 1], [0, h - 1]]).reshape(-1, 1, 2)
    warped = cv2.perspectiveTransform(corners, warp).reshape(-1, 2)
    area = abs(cv2.contourArea(warped.astype(np.float32)))
    src_area = max(1.0, float(w * h))
    scale = area / src_area
    if scale < ECC_MIN_SCALE or scale > ECC_MAX_SCALE:
        return False
    drift = np.linalg.norm(warped - corners.reshape(-1, 2), axis=1)
    max_allowed = ECC_MAX_CORNER_DRIFT_FRAC * max(h, w)
    if float(drift.max()) > max_allowed:
        return False
    return True

def ecc_homography_score(reference_img, moving_img):
    ref_gray = prepare_ecc_gray(reference_img)
    mov_gray = prepare_ecc_gray(moving_img)
    if ref_gray.shape != mov_gray.shape:
        mov_gray = cv2.resize(mov_gray, (ref_gray.shape[1], ref_gray.shape[0]), interpolation=cv2.INTER_AREA)
    baseline_mse = float(np.mean((ref_gray - mov_gray) ** 2))

    warp = np.eye(3, 3, dtype=np.float32)
    criteria = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, ECC_MAX_ITERS, ECC_EPS)
    try:
        cc, warp = cv2.findTransformECC(ref_gray, mov_gray, warp, cv2.MOTION_HOMOGRAPHY, criteria)
        if not ecc_warp_is_sane(warp, ref_gray.shape):
            return -baseline_mse
        warped = cv2.warpPerspective(
            mov_gray,
            warp,
            (ref_gray.shape[1], ref_gray.shape[0]),
            flags=cv2.INTER_LINEAR + cv2.WARP_INVERSE_MAP,
        )
        mse = float(np.mean((ref_gray - warped) ** 2))
        if mse > baseline_mse * ECC_MAX_WORSE_FACTOR:
            return -baseline_mse
        return float(cc) - mse
    except cv2.error:
        return -baseline_mse

def choose_best_duplicate(rows):
    if len(rows) == 1:
        return rows[0], {'duplicate_count': 1, 'representative_score': 1.0}

    images = []
    usable_rows = []
    for row in rows:
        path = os.path.join(ALIGNED_DIR, row['aligned_file'])
        img = cv2.imread(path, cv2.IMREAD_COLOR)
        if img is not None:
            usable_rows.append(row)
            images.append(img)

    if len(usable_rows) == 1:
        return usable_rows[0], {'duplicate_count': len(rows), 'representative_score': 1.0}
    if not usable_rows:
        return rows[0], {'duplicate_count': len(rows), 'representative_score': float('nan')}

    scores = []
    for i, ref_img in enumerate(images):
        pair_scores = []
        for j, mov_img in enumerate(images):
            if i == j:
                continue
            pair_scores.append(ecc_homography_score(ref_img, mov_img))
        scores.append(float(np.mean(pair_scores)) if pair_scores else 1.0)

    best_idx = int(np.nanargmax(scores))
    return usable_rows[best_idx], {
        'duplicate_count': len(rows),
        'representative_score': scores[best_idx],
        'candidate_scores_json': json.dumps(scores),
    }

def run_filtered_alignment(aligned_df):
    if REDO_FILTERED_ALIGNMENT:
        print('REDO_FILTERED_ALIGNMENT=True; deleting previous filtered aligned outputs.')
        ensure_empty_dir(FILTERED_DIR)
        if os.path.exists(FILTERED_CSV):
            os.remove(FILTERED_CSV)
    elif filtered_complete():
        print('Filtered aligned set already complete; skipping.')
        return pd.read_csv(FILTERED_CSV)
    else:
        os.makedirs(FILTERED_DIR, exist_ok=True)

    if aligned_df.empty:
        out_df = pd.DataFrame()
        out_df.to_csv(FILTERED_CSV, index=False)
        return out_df

    records = []
    group_cols = ['location', 'time_of_day', 'weather']
    for key, group in tqdm(list(aligned_df.groupby(group_cols, sort=True)), desc='Filter duplicates'):
        location, tod, weather = key
        rows = [row for _, row in group.iterrows()]
        selected, stats = choose_best_duplicate(rows)

        src_path = os.path.join(ALIGNED_DIR, selected['aligned_file'])
        img = cv2.imread(src_path, cv2.IMREAD_COLOR)
        if img is None:
            print(f"[MISS] filtered source missing: {src_path}")
            continue

        location_dir = os.path.join(FILTERED_DIR, clean_stem(location))
        os.makedirs(location_dir, exist_ok=True)
        out_name = f"{clean_stem(location)}_{clean_stem(tod)}_{clean_stem(weather)}.jpg"
        out_path = os.path.join(location_dir, out_name)
        cv2.imwrite(out_path, img, [cv2.IMWRITE_JPEG_QUALITY, JPEG_QUALITY])

        rec = dict(selected)
        rec.update(stats)
        rec['filtered_file'] = os.path.relpath(out_path, FILTERED_DIR)
        rec['selected_from_aligned_file'] = selected['aligned_file']
        records.append(rec)

    out_df = pd.DataFrame(records)
    out_df.to_csv(FILTERED_CSV, index=False)
    print(f'\nDone. Saved {len(out_df)} filtered rows to {FILTERED_CSV}')
    return out_df

filtered_df = run_filtered_alignment(aligned_df)
display(filtered_df.head())

In [ ]:
# Quick visual check for one filtered location.
if len(filtered_df):
    sample_location = filtered_df['location'].iloc[0]
    sample = filtered_df[filtered_df['location'] == sample_location].head(4)
    fig, axes = plt.subplots(1, len(sample), figsize=(5 * len(sample), 5))
    if len(sample) == 1:
        axes = [axes]
    for ax, (_, row) in zip(axes, sample.iterrows()):
        img = cv2.imread(os.path.join(FILTERED_DIR, row['filtered_file']), cv2.IMREAD_COLOR)
        ax.imshow(bgr_to_rgb(img))
        ax.set_title(row['filtered_file'])
        ax.axis('off')
    plt.show()
else:
    print('No filtered outputs to preview.')